## Feature Descriptives
Before the correlational and regression analyses in later notebooks, we describe the
article-level features in our analytic sample: how citation impact (raw counts, group
h-index, FWCI) differs between sharing and non-sharing articles, and then the distribution
of each remaining feature individually.

## Setup
The analytic sample is rebuilt (or loaded from the parquet cache) by `helpers.dataset.load_or_build()`, so this notebook runs standalone from a cold kernel.

In [1]:
import numpy as np
import pandas as pd
from scipy import stats
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio

from helpers.config import *
from helpers import dataset
from helpers.plotting import save_figure

pio.renderers.default = "browser"

In [2]:
combined, FEATURES_DF = dataset.load_or_build()

Loaded cached dataset from C:\Users\nirjo\Documents\University\PhD\Projects\eye_movement_data_sharing\.claude\worktrees\eye-movement-analysis-review-22482b\data_store (234 rows)


In [3]:
FINAL_DATASET_SIZE = len(combined)

---
## Per-Feature Descriptives
We now examine each remaining article feature individually: its distribution, and (where
relevant) a summary figure.

### Data Sharing Class
The Godwin et al. (2025) dataset tags each article by the type of data they share:
participant-level summaries (_PARTICIPANT_), trial-level data (_TRIAL_), fixation-level data
(_FIXATION_), or no data (_NONE_).<br>
Each article may share multiple types of data, but for our analysis we will focus on the
_highest granularity_ of shared data, in the order of FIXATION > TRIAL > PARTICIPANT > NONE.<br>

In [4]:
print(f"Final number of publications:\t {combined.shape[0]}")
print(f"Articles sharing any materials:\t{combined['is_sharing_anything'].sum()}")
print(f"Articles sharing data:\t{combined['is_sharing_data'].sum()}")

multi_sharing = (
    (combined[["BY_FIXATION", "BY_TRIAL", "BY_PPT"]] == "YES")
    .value_counts(dropna=False)
    .sort_index()
)
multi_sharing.index = multi_sharing.index.rename(
    {"BY_FIXATION": "FIXATION", "BY_TRIAL": "TRIAL", "BY_PPT": "PARTICIPANT"}
)

multi_sharing

Final number of publications:	 234
Articles sharing any materials:	83
Articles sharing data:	80


FIXATION  TRIAL  PARTICIPANT
False     False  False          154
                 True            23
          True   False           20
                 True             6
True      False  False           17
                 True             2
          True   False            9
                 True             3
Name: count, dtype: int64

### Publication Age
We measure publication age as weeks from an article's publication date through the end of the latest calendar year that every article in the sample has complete data for (see `helpers.dataset.weeks_through_shared_year`). This is preferred over measuring to each article's own OpenAlex `LastUpdate` census timestamp (precomputed and stored in our records as `Pub2UpdateTime`): census timestamps differ between articles within a single fetch, so `Pub2UpdateTime` is not comparable across articles.<br>
Alternatively, we can look directly at the publication year, which is either stored as `YEAR_PUBLISHED` in the Godwin dataset or as `PublicationYear` in the OpenAlex metadata (with some discrepancies between the two, see above).

In [5]:
publication_age_in_weeks, _ = dataset.weeks_through_shared_year(combined)
publication_age_in_weeks = publication_age_in_weeks.rename("AgeInWeeks")
print(f"Num records with missing age: {publication_age_in_weeks.isna().sum()} ({100 * publication_age_in_weeks.isna().sum() / FINAL_DATASET_SIZE :.1f}%)")
publication_age_in_weeks.describe().to_frame().T

Num records with missing age: 0 (0.0%)


,count,mean,std,min,25%,50%,75%,max
AgeInWeeks,234.0,327.384005,89.160977,161.0,249.178571,324.428571,394.821429,497.428571


In [6]:
age_fig = make_subplots(
    rows=2, cols=1, shared_xaxes=False, shared_yaxes=False,
    subplot_titles=["Publication Year", "Publication Age (Weeks)",],
    vertical_spacing=0.15,
)

age_fig.add_trace(
    row=1, col=1, trace=go.Histogram(
        name="Publication Year",
        x=combined["PublicationYear"],
        marker=dict(color="#ff7f0e"),
        xbins=dict(
            start=combined["PublicationYear"].min() - 0.5,
            end=combined["PublicationYear"].max() + 0.5,
            size=1
        )
))
age_fig.add_trace(
    row=2, col=1, trace=go.Histogram(
        name="Publication Age (Weeks)",
        x=publication_age_in_weeks,
        marker=dict(color="#1f77b4"),
        nbinsx=20,
))

# update layout and styling
age_fig.update_xaxes(
    row=1, col=1,
    # title=dict(text="year", font=AXIS_TITLE_FONT, standoff=2),
    tickmode='linear',
)
age_fig.update_xaxes(
    row=2, col=1,
    # title=dict(text="weeks", font=AXIS_TITLE_FONT, standoff=2),
    autorange="reversed",
)

age_fig.update_yaxes(title=dict(text="Count", font=AXIS_TITLE_FONT, standoff=2))

age_fig.update_layout(
    width=800, height=600,
    showlegend=False,
    margin=dict(t=60, b=40, l=40, r=20),
    template="simple_white",
)

if False:
    age_fig.show()

### Publication Counts per Author/Venue
We check how many different first- and last-authors there are in our dataset, and how many different articles each has published.<br>
Similarly, we count the number of unique publication venues (journals/conferences) and how many different articles were published in each.

In [7]:
authors = combined["Authors"].dropna().map(lambda auths: auths.split("; "))
print(f"Articles with missing authorship: {combined["Authors"].isna().sum()} ({100 * combined["Authors"].isna().sum() / FINAL_DATASET_SIZE :.1f}%)")
num_missing_venue = combined["VenueName"].isna().sum() + (combined["VenueName"].map(len) < 1).sum()
print(f"Num records with missing/invalid venue: {num_missing_venue} ({100 * num_missing_venue / FINAL_DATASET_SIZE :.1f}%)")

print("\n### First Authors: ###")
first_author = authors.map(lambda auths: auths[0]).replace(
    # manually validated matches
    {"York, A": "York, AA"}
).rename("first author")
first_counts = first_author.value_counts().rename("FirstAuthor")
num_articles_with_repeated_firsts = first_counts[first_counts > 1].sum()
print(f"Unique first-authors: {len(first_counts)}")
print(f"Articles by first-authors with multiple publications in the dataset: {num_articles_with_repeated_firsts} ({100 * num_articles_with_repeated_firsts / FINAL_DATASET_SIZE :.1f}%)")

print("\n### Last Authors: ###")
last_author = authors.map(lambda auths: auths[-1]).replace(
    # manually validated matches
    {"Becker, S": "Becker, SI", "Hooge, I": "Hooge, ITC", "Jiang, YV" : "Jiang, YHV", "Zelisky, G" : "Zelinsky, GJ"}
).rename("last author")
last_counts = last_author.value_counts().rename("LastAuthor")
num_articles_with_repeated_lasts = last_counts[last_counts > 1].sum()
print(f"Unique last-authors: {len(last_counts)}")
print(f"Articles by last-author with multiple publications in the dataset: {num_articles_with_repeated_lasts} ({100 * num_articles_with_repeated_lasts / FINAL_DATASET_SIZE :.1f}%)")

print("\n### Venue: ###")
articles_per_venue = combined["VenueName"].value_counts().sort_values(ascending=False).rename("Venue")
num_articles_with_repeated_venue = articles_per_venue[articles_per_venue > 1].sum()
print(f"Unique venues:\t{len(articles_per_venue.index)}")
print(f"Articles in venues with multiple publications in the dataset: {num_articles_with_repeated_venue} ({100 * num_articles_with_repeated_venue / FINAL_DATASET_SIZE :.1f}%)")

qs = [0.25, 0.50, 0.75, 0.90, 0.95]
pd.concat([first_counts.describe(qs), last_counts.describe(qs), articles_per_venue.describe(qs)], axis=1).T

Articles with missing authorship: 0 (0.0%)
Num records with missing/invalid venue: 0 (0.0%)

### First Authors: ###
Unique first-authors: 198
Articles by first-authors with multiple publications in the dataset: 66 (28.2%)

### Last Authors: ###
Unique last-authors: 167
Articles by last-author with multiple publications in the dataset: 105 (44.9%)

### Venue: ###
Unique venues:	23
Articles in venues with multiple publications in the dataset: 227 (97.0%)


,count,mean,std,min,25%,50%,75%,90%,95%,max
FirstAuthor,198.0,1.181818,0.469661,1.0,1.0,1.0,1.0,2.0,2.0,4.0
LastAuthor,167.0,1.401198,0.931768,1.0,1.0,1.0,1.0,2.0,3.0,7.0
Venue,23.0,10.173913,11.035695,1.0,1.0,7.0,15.5,21.2,31.9,43.0


### Article-Count Distribution Figure
We plot the following distributions:
1. Number of articles per first-author
2. Number of articles per last-author
3. Number of articles per venue

In [8]:
articles_count_distributions_fig = make_subplots(
    rows=2, cols=2, shared_xaxes=False, shared_yaxes=True,
    specs=[[{}, {}], [{"colspan": 2}, None]],
    subplot_titles=["First Authors", "Last Authors", "Venues"],
    vertical_spacing=0.3, horizontal_spacing=0.1,
)

# number of articles per first-author
articles_count_distributions_fig.add_trace(
    go.Histogram(
        name="First Authors",
        x=first_counts, xbins=dict(start=0.5, end=first_counts.max() + 0.5, size=1),
        marker=dict(color='#1b9e77'),
    ),
    row=1, col=1
)

# number of articles per last-author
articles_count_distributions_fig.add_trace(
    go.Histogram(
        name="Last Authors",
        x=last_counts, xbins=dict(start=0.5, end=last_counts.max() + 0.5, size=1),
        marker=dict(color='#d95f02'),
    ),
    row=1, col=2
)

# number of articles per venue
venue_counts = articles_per_venue.value_counts().sort_index()
articles_count_distributions_fig.add_trace(
    go.Bar(
        name="Venues",
        x=venue_counts.index, y=venue_counts,
        marker=dict(color='#7570b3'),
    ),
    row=2, col=1
)

# update layout and styling
articles_count_distributions_fig.update_xaxes(
    title=dict(text="Number of Articles Published", font=AXIS_TITLE_FONT),
    tickfont=AXIS_TICK_FONT,
)
articles_count_distributions_fig.update_yaxes(
    title=dict(text="Number of Authors", font=AXIS_TITLE_FONT),
    tickfont=AXIS_TICK_FONT,
    row=1, col=1
)
articles_count_distributions_fig.update_yaxes(
    title=dict(text="Number of Venues"),
    row=2, col=1
)

articles_count_distributions_fig.update_layout(
    width=700, height=500,
    title=dict(text="Number of Published Articles", font=TITLE_FONT),
    showlegend=False,
    margin=dict(t=60, b=0, l=0, r=0),
    template="simple_white", # Clean background for academic plots
)

if False:
    articles_count_distributions_fig.show()

### Authors per Publication
We check the distribution of authour-counts in each publication.

In [9]:
num_missing_auth_count = combined['NumAuthors'].isna().sum() + (combined['NumAuthors'] < 1).sum()
print(f"Number of articles with missing or invalid author counts: {num_missing_auth_count} ({100 * num_missing_auth_count / FINAL_DATASET_SIZE :.1f}%)")

combined["NumAuthors"].describe().to_frame().T

Number of articles with missing or invalid author counts: 0 (0.0%)


,count,mean,std,min,25%,50%,75%,max
NumAuthors,234.0,3.504274,1.594647,1.0,2.0,3.0,4.0,10.0


In [10]:
auths_per_article = combined["NumAuthors"].value_counts().sort_index()

authors_per_article_fig = go.Figure()
authors_per_article_fig.add_trace(go.Bar(
    x=auths_per_article.index, y=auths_per_article
))

# update layout and styling
authors_per_article_fig.update_xaxes(
    title=dict(text="Number of Authors per Articles", font=AXIS_TITLE_FONT),
    tickfont=AXIS_TICK_FONT,
    tickmode='linear', dtick=1
)
authors_per_article_fig.update_yaxes(
    title=dict(text="Number of Articles", font=AXIS_TITLE_FONT),
    tickfont=AXIS_TICK_FONT
)

authors_per_article_fig.update_layout(
    width=600, height=300,
    title=dict(text="Number of Authors per Article", font=TITLE_FONT),
    showlegend=False,
    margin=dict(t=40, b=0, l=0, r=0),
    template="simple_white", # Clean background for academic plots
)

if False:
    authors_per_article_fig.show()

### Impact Score Distribution
We check the distribution of the Impact-Factor heuristic we chose, which is either `Venue2yrMeanCitedness`, `VenueHIndex`, or `VenueI10Index`.

In [11]:
impact = combined[VENUE_IMPACT_METRIC]
num_missing_impact = impact.isna().sum() + (impact < 0).sum()
print(f"Num records missing impact score: {num_missing_impact} ({100 * num_missing_impact / FINAL_DATASET_SIZE :.1f}%)")

qs = [0.25, 0.50, 0.75, 0.90, 0.95]
combined[VENUE_IMPACT_METRIC].describe(qs).to_frame().T

Num records missing impact score: 0 (0.0%)


,count,mean,std,min,25%,50%,75%,90%,95%,max
Venue2yrMeanCitedness,234.0,2.334266,1.266142,0.429831,1.542299,1.807383,3.399298,3.611215,4.987941,6.054209


In [12]:
impact_score_distribution_fig = go.Figure()
impact_score_distribution_fig.add_trace(go.Violin(
    x=impact, name="",
    side="positive", fillcolor='#7570b3', line_color='black',
    points="all", pointpos=0.5, marker=dict(color='#bc8ee0'),
))

# update layout and styling
impact_score_distribution_fig.update_xaxes(
    title=dict(text=VENUE_IMAPCT_METRIC_NAME, font=AXIS_TITLE_FONT),
    tickfont=AXIS_TICK_FONT,
    tickmode='linear', dtick=1
)
impact_score_distribution_fig.update_yaxes(
    title=dict(text=None, font=AXIS_TITLE_FONT),
    tickfont=AXIS_TICK_FONT
)

impact_score_distribution_fig.update_layout(
    width=600, height=300,
    title=dict(text="Impact Score Distribution", font=TITLE_FONT),
    showlegend=False,
    margin=dict(t=40, b=0, l=0, r=0),
    template="simple_white", # Clean background for academic plots
)

if False:
    impact_score_distribution_fig.show()

### Feature Transformations
Some of these continuous features feed into log-transformed versions for the correlation and
regression analyses in notebook 03. We check for normality and skewness here, and apply a
transformation only where it improves the distribution. The three continuous features are
treated differently, for different reasons:
- **Publication age** (weeks since publication) is $log$-transformed on *functional* rather than distributional grounds: citation accumulation is multiplicative, so a $log$-scale linearises the age-citation relationship. This follows the recommendations of `Thelwall and Wilson (2014)` and `Clauset et al. (2009)`.
- **Number of authors** is $log$-transformed on *distributional* grounds: it is strongly right-skewed in its natural scale, and the $log$-transform renders it close to symmetric.
- **Venue impact** (2-year mean citedness) is kept in its **natural scale**: it is only mildly skewed, and a $log$-transform over-corrects it into a pronounced *left* skew. See the skewness check below.

_References:_
- Clauset, A., Shalizi, C. R., & Newman, M. E. (2009). Power-law distributions in empirical data. SIAM review, 51(4), 661-703.
- Thelwall, M., & Wilson, P. (2014). Regression for citation data: An evaluation of different methods. Journal of Informetrics, 8(4), 963-971.

### Skewness
Check the skewness of continuous features with and without $log$-transformation. A skewness value greater than 1 or less than -1 indicates a highly skewed distribution, while values between -1 and 1 indicate a moderately skewed distribution.

In [13]:
print(f"NumAuthors skewness:\tNatural Scale: {stats.skew(combined["NumAuthors"]) :.4f}\tLog Scale: {stats.skew(np.log(combined["NumAuthors"])) :.4f}\tLog+1 Scale: {stats.skew(np.log(combined["NumAuthors"] + 1)) :.4f}")
print(f"{VENUE_IMPACT_METRIC} skewness:\tNatural Scale {stats.skew(combined[VENUE_IMPACT_METRIC]) :.4f}\tLog Scale: {stats.skew(np.log(combined[VENUE_IMPACT_METRIC])) :.4f}\tLog+1 Scale: {stats.skew(np.log(combined[VENUE_IMPACT_METRIC] + 1)) :.4f}")

NumAuthors skewness:	Natural Scale: 1.3891	Log Scale: 0.1419	Log+1 Scale: 0.4294
Venue2yrMeanCitedness skewness:	Natural Scale 0.3783	Log Scale: -0.9655	Log+1 Scale: -0.4567


### Venue & Impact Score

In [14]:
# number of sharing/non-sharing articles per venue
venue_stats = (
    combined[["VenueName", "is_sharing_data"]]
    .dropna()
    .value_counts()
    .reset_index(drop=False)
)
venue_stats["share_label"] = venue_stats["is_sharing_data"].map({True: "SHARING", False: "NOT SHARING"})
venue_stats[VENUE_IMPACT_METRIC] = venue_stats["VenueName"].map(
    lambda venue: combined.loc[combined["VenueName"] == venue, VENUE_IMPACT_METRIC].iloc[0]
)
venue_stats["_total_articles"] = venue_stats["VenueName"].map(venue_stats.groupby("VenueName")["count"].sum())
venue_stats.sort_values(by=["_total_articles", VENUE_IMPACT_METRIC, "VenueName"], inplace=True, kind="stable", ascending=False)
venue_stats.drop(columns=["is_sharing_data", "_total_articles"], inplace=True)

layout = go.Layout(
    yaxis=dict(), xaxis=dict(), xaxis2=dict(overlaying='x', side='top', anchor='y')
)
venue_and_impact_fig = go.Figure(layout=layout)

# stacked bar trace of num sharing/non-sharing articles per venue
for sl in venue_stats["share_label"].unique():
    subset = venue_stats[venue_stats["share_label"] == sl]
    color = CLASS_COLORS[sl]
    venue_and_impact_fig.add_trace(go.Bar(
        name=f"{sl.title()} Data",
        legendgroup="Article Counts", showlegend=True,
        y=subset["VenueName"], x=subset["count"],
        orientation='h',
        marker=dict(color=color),
        xaxis='x'
    ))

# scatter plot of venue impact scores
venue_scores = venue_stats[["VenueName", VENUE_IMPACT_METRIC]].drop_duplicates()
venue_and_impact_fig.add_trace(go.Scatter(
    name=VENUE_IMAPCT_METRIC_NAME, legendgroup=VENUE_IMAPCT_METRIC_NAME, showlegend=True,
    y=venue_scores["VenueName"],
    x=venue_scores[VENUE_IMPACT_METRIC],
    mode='markers',
    marker=dict(color="#ff7f0e", size=8, symbol='diamond'),
    xaxis='x2'
))

# update layout and styling
venue_and_impact_fig.update_xaxes(
    selector=0,
    title=dict(text="Number of Articles", font=AXIS_TITLE_FONT, standoff=2),
    tickfont=AXIS_TICK_FONT
)
venue_and_impact_fig.update_xaxes(
    selector=1,
    title=dict(text=VENUE_IMAPCT_METRIC_NAME, font=AXIS_TITLE_FONT, standoff=2),
    tickfont=AXIS_TICK_FONT
)
venue_and_impact_fig.update_yaxes(
    title=dict(font=AXIS_TITLE_FONT, standoff=2),
    tickfont=AXIS_TICK_FONT,
    categoryorder="array", categoryarray=venue_stats["VenueName"].drop_duplicates().tolist()
)
venue_and_impact_fig.update_layout(
    width=1200, height=600,
    title=dict(
        text="Distribution of Publications and Journal Impact", font=TITLE_FONT,
        xref="container", xanchor="center", x=0.5,
        yref="container", yanchor="bottom", y=0.975
    ),
    barmode='stack',
    legend=dict(
        visible=False, orientation="h",
        yanchor="bottom", y=-0.23, yref="paper",
        xanchor="left", x=0.0, xref="paper",
        font=AXIS_TICK_FONT,
    ),
    margin=dict(t=75, b=5, l=250, r=5),
    template="simple_white",
)

if False:
    venue_and_impact_fig.show()

In [15]:
# flip to True to export this figure into `output/`
if False:
    save_figure(venue_and_impact_fig, "venue_impact.png", width=1200, height=600)

### Has American Author + Is Open-Access + Has Pre-Print
We check the rate of these three binary features in our dataset.

In [16]:
binaries = {'HasUSAuthor': 'Has US Author', 'IsOpenAccess': 'Is OA', 'HasPreprint': 'Has Pre-Print'}
for b in binaries:
    bin_name = binaries[b]
    print(f"### {bin_name} ###")
    num_missing_binary = combined[b].isna().sum()
    print(f"Num records missing `{bin_name}`: {num_missing_binary}")
    percent_binary = combined[combined[b]].shape[0] * 100 / FINAL_DATASET_SIZE
    print(f"% `{bin_name}`: {combined[b].sum()} artciles ({percent_binary:.1f}%)")
    print()

### Has US Author ###
Num records missing `Has US Author`: 0
% `Has US Author`: 99 artciles (42.3%)

### Is OA ###
Num records missing `Is OA`: 0


% `Is OA`: 199 artciles (85.0%)

### Has Pre-Print ###
Num records missing `Has Pre-Print`: 0
% `Has Pre-Print`: 47 artciles (20.1%)



In [17]:
binary_counts = pd.concat([combined[b].value_counts().rename(b) for b in binaries.keys()], axis=1)
binary_percents = (100 * binary_counts / binary_counts.sum(axis=0)).round(2).T

binary_percents_fig = go.Figure()
for b in binary_percents.columns:
    binary_percents_fig.add_trace(go.Bar(
        x=binary_percents.index,
        y=binary_percents[b],
        name=b, marker_color="green" if b else "red"
    ))

# update layout and styling
binary_percents_fig.update_xaxes(
    title=dict(text="Binary Feature", font=AXIS_TITLE_FONT),
    tickfont=AXIS_TICK_FONT,
)
binary_percents_fig.update_yaxes(
    title=dict(text="% of Articles", font=AXIS_TITLE_FONT),
    tickfont=AXIS_TICK_FONT
)

binary_percents_fig.update_layout(
    width=600, height=400,
    title=dict(text="Binary Features", font=TITLE_FONT),
    barmode='stack',
    showlegend=False,
    margin=dict(t=40, b=0, l=0, r=0),
    template="simple_white", # Clean background for academic plots
)

if False:
    binary_percents_fig.show()